# *Fine-tuning* de XLM-T, RoBERTuito y BETO (ticket #33)

Ajusta los modelos Transformer sobre las particiones congeladas del #29 y deja, por corrida:
un JSON en `resultados/` (formato del #29, con pérdida y métricas por época), las curvas en
`figuras/` (PNG y PDF) y los pesos en Drive o en Hugging Face Hub (privado).

| Corrida | Modelo | Entrenamiento |
|---|---|---|
| `xlmt` | `cardiffnlp/twitter-xlm-roberta-base` | FakeDeS |
| `xlmt-2etapas` | `cardiffnlp/twitter-xlm-roberta-base` | LIAR + FakeNewsNet, después FakeDeS |
| `robertuito` | `pysentimiento/robertuito-base-uncased` | FakeDeS, con `preprocess_tweet` de pysentimiento |
| `beto` | `dccuchile/bert-base-spanish-wwm-cased` | FakeDeS |
| `beto-chequeado` | `dccuchile/bert-base-spanish-wwm-cased` | FakeDeS, después afirmaciones de Chequeado (#36) |

La época se elige por F1 macro en la validación de FakeDeS; la prueba de FakeDeS se evalúa
una vez, con el modelo elegido. El corpus argentino no se carga nunca.

`beto-chequeado` sigue entrenando el BETO de FakeDeS sobre `afirmaciones_chequeado.csv`
(títulos de notas de Chequeado, sin las notas ni los tuits del corpus de prueba) y elige la
época de esa etapa por F1 macro en su validación. Es la adaptación posterior a la evaluación
del corpus argentino: el BETO de FakeDeS no generalizó a publicaciones cortas.

Corre en **Colab** o en **Kaggle**; detecta solo dónde está.

- **Colab:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*, y después *Ejecutar todas*.
  Al final descarga un `.zip` con `resultados/` y `figuras/`; los pesos quedan en Drive.
- **Kaggle** (cuenta verificada con teléfono): *Settings → Accelerator → GPU T4 x2* e
  *Internet → On*, y después *Save Version → Save & Run All*, que corre en segundo plano.
  El `.zip` y los pesos quedan en la pestaña *Output*. Se usa una sola de las dos GPU, para que
  el lote efectivo siga siendo 16.

**Largo máximo.** La mitad de las noticias de FakeDeS pasa los 500 *tokens* (mediana: 503 con
XLM-T, 474 con BETO, 447 con RoBERTuito). XLM-T y BETO usan 512, su límite. RoBERTuito tiene
130 posiciones, así que se corta en 128 y trunca el 97 % de las noticias: es una limitación
del modelo y se reporta. El truncado es por la derecha: se conservan el titular y el primer
tramo del cuerpo. LIAR + FakeNewsNet (afirmaciones y titulares, percentil 99 = 50 *tokens*) usa 128.

In [ ]:
RAMA = "main"          # rama del repo de la que se toma el módulo
CORRIDAS = ["beto-chequeado"]   # todas: ["xlmt", "xlmt-2etapas", "robertuito", "beto", "beto-chequeado"]
SEMILLA = 42
GUARDAR_PESOS = "carpeta"   # "carpeta" (Drive en Colab, Output en Kaggle) | "hub" | "no"
CARPETA_DRIVE = "/content/drive/MyDrive/PFI/clasificador"
USUARIO_HUB = "martinnbejarano"   # repos privados <usuario>/pfi-<id_corrida>; token en el secreto HF_TOKEN

HIPERPARAMETROS = {
    "tasa_aprendizaje": 2e-5,
    "epocas": 5,                  # FakeDeS; se elige la mejor por F1 macro en validación
    "epocas_etapa_previa": 2,     # LIAR + FakeNewsNet
    "lote": 8,
    "acumulacion_gradiente": 2,   # lote efectivo 16
    "calentamiento": 0.1,         # fracción de pasos con tasa creciente
    "decaimiento_pesos": 0.01,
    "max_length_etapa_previa": 128,
    "epocas_chequeado": 5,        # se elige la mejor por F1 macro en la validación de Chequeado
    "max_length_chequeado": 128,  # afirmaciones: mediana de 13 palabras
}

# Modo humo: modelo diminuto, pocas filas, una época, sin GPU. Nada se guarda en resultados/.
HUMO = False

In [ ]:
import os
import sys

EN_COLAB = "google.colab" in sys.modules
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if EN_KAGGLE:
    # Kaggle da dos T4; con las dos, Trainer duplicaría el lote efectivo. Tiene que ir antes de importar torch.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if EN_COLAB or EN_KAGGLE:
    # Ambos traen torch, transformers, accelerate, pandas y scikit-learn.
    !pip install -q pysentimiento openpyxl
    # En Kaggle se clona fuera de /kaggle/working, para que la salida no incluya el repositorio.
    %cd {"/tmp" if EN_KAGGLE else "/content"}
    !git clone -q --depth 1 -b {RAMA} https://github.com/martinnbejarano/PFI-Wiki.git
    %cd PFI-Wiki/prototipo/clasificador
SALIDA = "/kaggle/working" if EN_KAGGLE else "."
CARPETA_PESOS = CARPETA_DRIVE if EN_COLAB else f"{SALIDA}/pesos"

In [ ]:
import hashlib
import math
import shutil
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from pysentimiento.preprocessing import preprocess_tweet
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                          TrainingArguments, set_seed)

from datos import AFIRMACIONES_CHEQUEADO, cargar_chequeado, cargar_todos, congelar, descargar, preparar, preprocesar
from resultados import construir_resultado, guardar, metricas

assert HUMO or torch.cuda.is_available(), "Falta la GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4."

corpus = preparar(cargar_todos(descargar("datos")), SEMILLA)
# Falla si no coincide con `particiones.csv.gz`: todos los modelos ven los mismos ejemplos.
HASH_PARTICIONES = congelar(corpus, "particiones.csv.gz")
assert set(corpus["dataset"]) == {"liar", "fakenewsnet", "fakedes"}   # el corpus argentino no entra
# Las afirmaciones de Chequeado traen su propia partición (sin prueba) y su propia huella.
HASH_CHEQUEADO = hashlib.sha256(AFIRMACIONES_CHEQUEADO.read_bytes()).hexdigest()
corpus = pd.concat([corpus, cargar_chequeado()], ignore_index=True)

if HUMO:
    corpus = corpus.groupby(["dataset", "particion", "etiqueta"]).head(8)
    HIPERPARAMETROS.update(epocas=1, epocas_etapa_previa=1, lote=4, acumulacion_gradiente=1)
    CARPETA_RESULTADOS = CARPETA_FIGURAS = Path(tempfile.mkdtemp(prefix="humo_"))
else:
    CARPETA_RESULTADOS, CARPETA_FIGURAS = Path("resultados"), Path("figuras")
CARPETA_FIGURAS.mkdir(exist_ok=True)

print("hash de las particiones:", HASH_PARTICIONES, "· Chequeado:", HASH_CHEQUEADO)
corpus.groupby(["dataset", "particion"])["etiqueta"].value_counts().unstack()

In [ ]:
XLMT, ROBERTUITO, BETO = "cardiffnlp/twitter-xlm-roberta-base", "pysentimiento/robertuito-base-uncased", "dccuchile/bert-base-spanish-wwm-cased"
CONFIGURACIONES = {
    "xlmt": {"modelo": "xlm-t", "checkpoint": XLMT, "max_length": 512, "etapa_previa": False, "preprocesamiento": "datos.preprocesar"},
    "xlmt-2etapas": {"modelo": "xlm-t", "checkpoint": XLMT, "max_length": 512, "etapa_previa": True, "preprocesamiento": "datos.preprocesar"},
    "robertuito": {"modelo": "robertuito", "checkpoint": ROBERTUITO, "max_length": 128, "etapa_previa": False, "preprocesamiento": "pysentimiento.preprocess_tweet(lang='es')"},
    "beto": {"modelo": "beto", "checkpoint": BETO, "max_length": 512, "etapa_previa": False, "preprocesamiento": "datos.preprocesar"},
    "beto-chequeado": {"modelo": "beto", "checkpoint": BETO, "max_length": 512, "etapa_previa": False, "preprocesamiento": "datos.preprocesar", "etapa_final": "chequeado"},
}
if HUMO:
    for config in CONFIGURACIONES.values():
        config.update(checkpoint="hf-internal-testing/tiny-random-RobertaForSequenceClassification", max_length=32)
    HIPERPARAMETROS["max_length_etapa_previa"] = 32

PREPROCESADORES = {"datos.preprocesar": preprocesar, "pysentimiento.preprocess_tweet(lang='es')": lambda t: preprocess_tweet(t, lang="es")}


def seleccion(datasets, particion):
    return corpus[corpus["dataset"].isin(datasets) & (corpus["particion"] == particion)]


def codificar(tokenizador, preprocesador, datos, max_length):
    # Truncado por la derecha: queda el titular y el comienzo del cuerpo.
    codificado = tokenizador([preprocesador(t) for t in datos["texto"]], truncation=True, max_length=max_length)
    return [
        {**{clave: valores[i] for clave, valores in codificado.items()}, "labels": int(etiqueta == "falso")}
        for i, etiqueta in enumerate(datos["etiqueta"])
    ]


def puntajes(logits):
    # Probabilidad de «falso» (etiqueta 1): el puntaje del Módulo 1.
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].tolist()


def metricas_de_epoca(prediccion):
    m = metricas(["falso" if e else "verdadero" for e in prediccion.label_ids], puntajes(prediccion.predictions))
    return {"f1_macro": m["f1_macro"], "auc_roc": m["auc_roc"]}


def entrenar(modelo, tokenizador, entrenamiento, validacion, epocas, carpeta):
    pasos = math.ceil(len(entrenamiento) / (HIPERPARAMETROS["lote"] * HIPERPARAMETROS["acumulacion_gradiente"])) * epocas
    argumentos = TrainingArguments(
        output_dir=carpeta,
        num_train_epochs=epocas,
        learning_rate=HIPERPARAMETROS["tasa_aprendizaje"],
        per_device_train_batch_size=HIPERPARAMETROS["lote"],
        per_device_eval_batch_size=HIPERPARAMETROS["lote"] * 2,
        gradient_accumulation_steps=HIPERPARAMETROS["acumulacion_gradiente"],
        warmup_steps=int(HIPERPARAMETROS["calentamiento"] * pasos),
        weight_decay=HIPERPARAMETROS["decaimiento_pesos"],
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,   # la época con mejor F1 macro en validación
        metric_for_best_model="f1_macro",
        save_total_limit=1,
        save_only_model=True,
        report_to="none",
        seed=SEMILLA,
    )
    entrenador = Trainer(
        model=modelo, args=argumentos, train_dataset=entrenamiento, eval_dataset=validacion,
        processing_class=tokenizador, data_collator=DataCollatorWithPadding(tokenizador), compute_metrics=metricas_de_epoca,
    )
    entrenador.train()
    return entrenador


def epocas_de(entrenador, etapa):
    historial = entrenador.state.log_history
    perdida = {round(h["epoch"]): h["loss"] for h in historial if "loss" in h}
    return [
        {
            "etapa": etapa,
            "epoca": round(h["epoch"]),
            "perdida_entrenamiento": perdida.get(round(h["epoch"])),
            "perdida_validacion": h["eval_loss"],
            "f1_macro_validacion": h["eval_f1_macro"],
            "auc_roc_validacion": h["eval_auc_roc"],
        }
        for h in historial
        if "eval_loss" in h
    ]


def evaluar(entrenador, datos_codificados, datos):
    return metricas(datos["etiqueta"].tolist(), puntajes(entrenador.predict(datos_codificados).predictions))

In [ ]:
def graficar(epocas, id_corrida):
    # Una sola serie de épocas; una línea vertical marca cada cambio de etapa.
    x = range(1, len(epocas) + 1)
    figura, (ejes_perdida, ejes_f1) = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
    ejes_perdida.plot(x, [e["perdida_entrenamiento"] for e in epocas], "o-", label="entrenamiento")
    ejes_perdida.plot(x, [e["perdida_validacion"] for e in epocas], "s--", label="validación")
    ejes_perdida.set(xlabel="época", ylabel="pérdida (entropía cruzada)")
    ejes_perdida.legend(frameon=False)
    ejes_f1.plot(x, [e["f1_macro_validacion"] for e in epocas], "o-", color="C2")
    ejes_f1.set(xlabel="época", ylabel="F1 macro en validación", ylim=(0, 1))
    cambios = [i for i in range(1, len(epocas)) if epocas[i]["etapa"] != epocas[i - 1]["etapa"]]
    for ejes in (ejes_perdida, ejes_f1):
        ejes.set_xticks(list(x))
        ejes.grid(alpha=0.3)
        ejes.spines[["top", "right"]].set_visible(False)
        for i in cambios:
            ejes.axvline(i + 0.5, color="grey", lw=0.8, ls=":")
    figura.suptitle(id_corrida, fontsize=10)
    for extension in ("png", "pdf"):
        figura.savefig(CARPETA_FIGURAS / f"{id_corrida}.{extension}", dpi=200)
    plt.show()


def guardar_pesos(modelo, tokenizador, resultado):
    carpeta = Path(tempfile.mkdtemp()) / resultado["id_corrida"]
    modelo.save_pretrained(carpeta)
    tokenizador.save_pretrained(carpeta)
    guardar(resultado, carpeta)   # el Space lee de acá la versión y el preprocesamiento
    if HUMO or GUARDAR_PESOS == "no":
        return carpeta
    if GUARDAR_PESOS == "carpeta":
        destino = Path(CARPETA_PESOS) / resultado["id_corrida"]
        shutil.copytree(carpeta, destino, dirs_exist_ok=True)
        return destino
    from huggingface_hub import HfApi

    if EN_KAGGLE:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

    repo = f"{USUARIO_HUB}/pfi-{resultado['id_corrida']}"
    HfApi().create_repo(repo, private=True, exist_ok=True)
    HfApi().upload_folder(repo_id=repo, folder_path=carpeta)
    return f"https://huggingface.co/{repo}"


if EN_COLAB and not HUMO and GUARDAR_PESOS == "carpeta":
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
tabla = []
for nombre in CORRIDAS:
    config = CONFIGURACIONES[nombre]
    set_seed(SEMILLA)   # también fija la inicialización de la cabeza de clasificación
    tokenizador = AutoTokenizer.from_pretrained(config["checkpoint"])
    modelo = AutoModelForSequenceClassification.from_pretrained(
        config["checkpoint"], num_labels=2, id2label={0: "verdadero", 1: "falso"}, label2id={"verdadero": 0, "falso": 1}
    )
    preprocesador = PREPROCESADORES[config["preprocesamiento"]]
    carpeta = tempfile.mkdtemp()   # checkpoints intermedios: no se versionan
    epocas = []

    if config["etapa_previa"]:
        previos = ["liar", "fakenewsnet"]
        entrenador = entrenar(
            modelo, tokenizador,
            codificar(tokenizador, preprocesador, seleccion(previos, "entrenamiento"), HIPERPARAMETROS["max_length_etapa_previa"]),
            codificar(tokenizador, preprocesador, seleccion(previos, "validacion"), HIPERPARAMETROS["max_length_etapa_previa"]),
            HIPERPARAMETROS["epocas_etapa_previa"], carpeta + "/previa",
        )
        epocas += epocas_de(entrenador, "liar+fakenewsnet")
        modelo = entrenador.model   # el mejor de la etapa previa (F1 macro en su validación)

    validacion, prueba = seleccion(["fakedes"], "validacion"), seleccion(["fakedes"], "prueba")
    validacion_codificada = codificar(tokenizador, preprocesador, validacion, config["max_length"])
    entrenador = entrenar(
        modelo, tokenizador,
        codificar(tokenizador, preprocesador, seleccion(["fakedes"], "entrenamiento"), config["max_length"]),
        validacion_codificada, HIPERPARAMETROS["epocas"], carpeta + "/fakedes",
    )
    epocas += epocas_de(entrenador, "fakedes")
    evaluaciones = {}
    etapa_final = config.get("etapa_final") or "fakedes"
    if etapa_final == "chequeado":
        validacion_chequeado = seleccion(["chequeado"], "validacion")
        validacion_chequeado_codificada = codificar(tokenizador, preprocesador, validacion_chequeado, HIPERPARAMETROS["max_length_chequeado"])
        entrenador = entrenar(
            entrenador.model, tokenizador,
            codificar(tokenizador, preprocesador, seleccion(["chequeado"], "entrenamiento"), HIPERPARAMETROS["max_length_chequeado"]),
            validacion_chequeado_codificada, HIPERPARAMETROS["epocas_chequeado"], carpeta + "/chequeado",
        )
        epocas += epocas_de(entrenador, "chequeado")
        evaluaciones["chequeado/validacion"] = evaluar(entrenador, validacion_chequeado_codificada, validacion_chequeado)
    elegida = max((e for e in epocas if e["etapa"] == etapa_final), key=lambda e: e["f1_macro_validacion"])["epoca"]

    entrenado_en = {"chequeado": "fakedes-chequeado"}.get(config.get("etapa_final"), "liar-fakenewsnet-fakedes" if config["etapa_previa"] else "fakedes")
    resultado = construir_resultado(
        id_corrida=f"{config['modelo']}_{entrenado_en}_s{SEMILLA}",
        modelo=config["modelo"],
        entrenado_en=entrenado_en,
        hiperparametros={
            **HIPERPARAMETROS, **config, "epoca_elegida": elegida,
            **({"hash_chequeado": HASH_CHEQUEADO} if etapa_final == "chequeado" else {}),
            "optimizador": "AdamW (Trainer)", "precision": "fp16" if torch.cuda.is_available() else "fp32",
        },
        evaluaciones={
            **evaluaciones,
            "fakedes/validacion": evaluar(entrenador, validacion_codificada, validacion),
            "fakedes/prueba": evaluar(entrenador, codificar(tokenizador, preprocesador, prueba, config["max_length"]), prueba),
        },
        semilla=SEMILLA,
        hash_particiones=HASH_PARTICIONES,
        epocas=epocas,
    )
    resultado["entorno"].update(
        torch=torch.__version__, transformers=transformers.__version__,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    )
    print(guardar(resultado, CARPETA_RESULTADOS))
    graficar(epocas, resultado["id_corrida"])
    print("pesos:", guardar_pesos(entrenador.model, tokenizador, resultado))
    for evaluacion, m in resultado["evaluaciones"].items():
        tabla.append({"corrida": resultado["id_corrida"], "evaluacion": evaluacion, "f1_macro": m["f1_macro"], "auc_roc": m["auc_roc"], "n": m["n"]})

    del modelo, entrenador
    shutil.rmtree(carpeta, ignore_errors=True)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(tabla).round(4)

In [ ]:
if (EN_COLAB or EN_KAGGLE) and not HUMO:
    !zip -q -r {SALIDA}/fine_tuning.zip resultados figuras
    if EN_COLAB:
        from google.colab import files

        files.download("fine_tuning.zip")